# NLP 特征提取与 KMeans 聚类训练

**目的：** 从 MySQL 读取招聘数据，提取 TF-IDF 文本特征，训练 KMeans 聚类模型，保存到 models/

In [ ]:
import pymysql
import pandas as pd
import numpy as np
import pickle
import os
import jieba
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# 中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

print('导入完成')

In [ ]:
# ====== 从 MySQL 读取数据 ======
conn = pymysql.connect(
    host='localhost', port=3306, user='root',
    password='123456', database='recruitment_ai', charset='utf8mb4'
)
df = pd.read_sql('SELECT * FROM job_listing', conn)
conn.close()
print(f'读取 {len(df)} 条数据')
df[['job_title', 'skills_raw', 'city', 'experience', 'education', 'industry']].head(3)

In [ ]:
# ====== 文本预处理 ======

# 加载停用词（如没有文件就用基础列表）
stopwords_path = '../data/stopwords.txt'
if os.path.exists(stopwords_path):
    with open(stopwords_path, 'r', encoding='utf-8') as f:
        stopwords = set(f.read().splitlines())
else:
    stopwords = {'的', '了', '和', '是', '就', '都', '而', '及', '与', '着', '或', '一个', '没有', '我们', '你们', '他们'}

def preprocess(text):
    """分词 + 去停用词 + 去非中文英文"""
    if pd.isna(text):
        return ''
    # 保留中英文数字
    text = re.sub(r'[^\u4e00-\u9fa5a-zA-Z0-9\s]', ' ', str(text))
    words = jieba.cut(text)
    return ' '.join(w for w in words if len(w) > 1 and w not in stopwords)

# 构造文本特征：技能关键词 + 岗位名称
df['text_feature'] = (df['skills_raw'].fillna('') + ' ' + df['job_title'].fillna('')).apply(preprocess)
print('文本预处理完成')
print(df['text_feature'].iloc[0][:200])

In [ ]:
# ====== TF-IDF 特征提取 ======
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2
)
X_tfidf = tfidf.fit_transform(df['text_feature'])
print(f'TF-IDF 特征矩阵: {X_tfidf.shape}')
print(f'词汇量: {len(tfidf.get_feature_names_out())}')

In [ ]:
# ====== 训练 KMeans 聚类 ======
N_CLUSTERS = 5
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_tfidf)
df['cluster_label'] = labels

print('聚类分布:')
for i in range(N_CLUSTERS):
    count = (labels == i).sum()
    print(f'  类别{i}: {count} 条 ({count/len(labels)*100:.1f}%)')

In [ ]:
# ====== PCA 可视化聚类结果 ======
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_tfidf.toarray())

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='viridis', alpha=0.6, s=10)
ax.set_title('KMeans 聚类结果 (PCA 降维)', fontsize=14)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
plt.colorbar(scatter, label='Cluster')
plt.tight_layout()
plt.show()

In [ ]:
# ====== 分析每个聚类的关键词 ======
feature_names = tfidf.get_feature_names_out()
for i in range(N_CLUSTERS):
    centroid = kmeans.cluster_centers_[i]
    top_idx = centroid.argsort()[-10:][::-1]
    top_words = [feature_names[j] for j in top_idx]
    print(f'类别{i}: {", ".join(top_words)}')

In [ ]:
# ====== 保存模型 ======
model_dir = '../models'
os.makedirs(model_dir, exist_ok=True)

# 保存 KMeans 模型
with open(f'{model_dir}/kmeans_model.pkl', 'wb') as f:
    pickle.dump(kmeans, f)

# 保存 TF-IDF 向量器（推理时需要用同样的方式转换文本）
with open(f'{model_dir}/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

# 保存聚类结果到 MySQL
conn = pymysql.connect(
    host='localhost', port=3306, user='root',
    password='123456', database='recruitment_ai', charset='utf8mb4'
)
cursor = conn.cursor()
for job_id, label in zip(df['id'], labels):
    cursor.execute(
        'INSERT INTO cluster_result (job_id, cluster_label) VALUES (%s, %s) ON DUPLICATE KEY UPDATE cluster_label=%s',
        (int(job_id), int(label), int(label))
    )
conn.commit()
conn.close()

print(f'模型已保存: {model_dir}/kmeans_model.pkl')
print(f'向量器已保存: {model_dir}/tfidf_vectorizer.pkl')
print(f'聚类结果已写入 MySQL cluster_result 表')